In [ ]:
import numpy as np
import h5py
import glob
import pandas as pd
from sklearn.model_selection import train_test_split

from keras.models import load_model
from keras.callbacks import LearningRateScheduler, ModelCheckpoint
from keras.preprocessing.image import ImageDataGenerator
from keras import backend as K
from keras.utils import np_utils
from keras.optimizers import SGD
from keras.layers import *
from keras.layers.core import Dense, Dropout, Activation
from keras.layers.convolutional import Convolution2D
from keras.layers.pooling import AveragePooling2D, GlobalAveragePooling2D, MaxPooling2D
from keras.layers.normalization import BatchNormalization
from keras.models import Model
import keras.backend as K
from keras.models import load_model

from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix
from sklearn.utils.multiclass import unique_labels

import MDAnalysis

import sys
sys.path.append('/home/agp2004/HDaT/ML/deep_trajedy')
from trajectory_tools.tools import *
from nets.densenet161_CYC import *

from vis.utils import utils
from vis.visualization import *

from matplotlib import pyplot as plt
%matplotlib inline

import os

# os.environ["CUDA_VISIBLE_DEVICES"]="2,3"
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
K.tensorflow_backend._get_available_gpus()

In [ ]:
df = pd.read_csv('/athena/hwlab/scratch/agp2004/MFSD2A/DNN/data/history_n1.csv')
ax1 = df[['val_loss', 'loss']].plot(kind='line', markevery=[df['val_loss'].idxmin()], marker='o', markerfacecolor='g')
ax2 = df[['val_acc', 'acc']].plot(kind='line', markevery=[df['val_loss'].idxmin()], marker='o', markerfacecolor='g')

ax1.legend(['validation loss', 'training loss'])
ax2.legend(['validation accuracy', 'training accuracy'])

ax1.set_title('0.8 ns stride')
ax2.set_title('0.8 ns stride')

ax1.set_xlabel('epoch')
ax2.set_xlabel('epoch')

ax1.set_ylabel('loss')
ax2.set_ylabel('accuracy')

ax1 = ax1.get_figure()
ax2 = ax2.get_figure()
# ax1.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/loss_2.png', dpi=300)
# ax2.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/acc_2.png', dpi=300)


In [ ]:

## user parameters ####################
class1_labels_file = '/athena/hwlab/scratch/agp2004/MFSD2A/DNN/data/occluded/labels.dat'
class1_data_file = '/athena/hwlab/scratch/agp2004/MFSD2A/DNN/data/occluded/coordinates.dat'
class2_labels_file = '/athena/hwlab/scratch/agp2004/MFSD2A/DNN/data/outward/labels.dat'
class2_data_file = '/athena/hwlab/scratch/agp2004/MFSD2A/DNN/data/outward/coordinates.dat'
checkpoint_stem = '/athena/hwlab/scratch/agp2004/MFSD2A/DNN/data/'
weights_file = '/athena/hwlab/scratch/agp2004/MFSD2A/DNN/data/_weights_stride_1.hdf5'

dim_diff = 10
num_classes = 2
stride=1
######################################

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
K.tensorflow_backend._get_available_gpus()

h5f1 = h5py.File(class1_data_file,'r')
CLASS1_train_set = h5f1['dataset_class1'][::stride]
h5f1.close()

h5f2 = h5py.File(class1_labels_file,'r')
class1_train_set = h5f2['labels_class1'][::stride]
h5f2.close()

h5f3 = h5py.File(class2_data_file,'r')
CLASS2_train_set = h5f3['dataset_class2'][::stride]
h5f3.close()

h5f4 = h5py.File(class2_labels_file,'r')
class2_train_set = h5f4['labels_class2'][::stride]
h5f4.close()

traj_xyz = np.concatenate((CLASS1_train_set, CLASS2_train_set), axis=0)
labels = np.concatenate((class1_train_set, class2_train_set))

traj = traj_xyz


In [ ]:

traj = traj_to_img(traj, dim_diff=dim_diff)
X_train, X_test, Y_train, Y_test = train_test_split(traj, labels, test_size=0.20, random_state=42)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train, Y_train, test_size=0.30, random_state=66)
Y_train = np_utils.to_categorical(Y_train, num_classes)
Y_valid = np_utils.to_categorical(Y_valid, num_classes)
Y_test = np_utils.to_categorical(Y_test, num_classes)

train_datagen = ImageDataGenerator(
    featurewise_center=True,
    featurewise_std_normalization=True
    )

train_datagen.fit(X_train)

X_train = train_datagen.standardize(X_train)
X_valid = train_datagen.standardize(X_valid)
X_test = train_datagen.standardize(X_test)


def makemodel(X_train, Y_train, X_valid, Y_valid):
    if __name__ == '__main__':

        img_rows, img_cols =  X_train.shape[1], X_train.shape[2] # Resolution of inputs
        channel = 3
        num_classes = 2
        batch_size = 6
        nb_epoch = 100
        lr=1e-4

        # Load our model
        model = densenet161_model(img_rows=img_rows, img_cols=img_cols, color_type=channel, num_classes=num_classes, nb_dense_block=3, growth_rate=48, nb_filter=96, reduction=0.5, dropout_rate=0.2, weight_decay=1e-4)

        def lr_schedule(epoch, lr):
            return lr * (0.1 ** int(epoch / 10))
    #     LearningRateScheduler(lr_schedule)

        return model

model = makemodel(X_train, Y_train, X_valid, Y_valid)

model.load_weights(weights_file)

scores = model.predict(X_test, verbose=1)
accuracy = 100*np.sum(np.argmax(scores, axis=1)==np.argmax(Y_test, axis=1))/Y_test.shape[0]
print("Overall test set accuracy is    %",accuracy)

class1_ind = np.where(np.argmax(Y_test, axis=1)==0)
scores = model.predict(X_test[class1_ind], verbose=1)
accuracy = 100*np.sum(np.argmax(scores, axis=1)==np.argmax(Y_test[class1_ind], axis=1))/Y_test[class1_ind].shape[0]
print("class1 test set accuracy is    %",accuracy)

class2_ind = np.where(np.argmax(Y_test, axis=1)==1)
scores = model.predict(X_test[class2_ind], verbose=1)
accuracy = 100*np.sum(np.argmax(scores, axis=1)==np.argmax(Y_test[class2_ind], axis=1))/Y_test[class2_ind].shape[0]
print("Class2 test set accuracy is    %",accuracy)


In [ ]:
class1_ind = np.where(np.argmax(Y_test, axis=1)==0)
scores = model.predict(X_test[class1_ind], verbose=1)
class1_sal = X_test[class1_ind][(np.argmax(scores, axis=1)==0)]
print(class1_sal.shape)

class2_ind = np.where(np.argmax(Y_test, axis=1)==1)
scores = model.predict(X_test[class2_ind], verbose=1)
class2_sal = X_test[class2_ind][(np.argmax(scores, axis=1)==1)]
print(class2_sal.shape)

In [ ]:
def find_layer_idx(model, layer_name):
    """Looks up the layer index corresponding to `layer_name` from `model`.
    Args:
        model: The `keras.models.Model` instance.
        layer_name: The name of the layer to lookup.
    Returns:
        The layer index if found. Raises an exception otherwise.
    """
    layer_idx = None
    for idx, layer in enumerate(model.layers):
        if layer.name == layer_name:
            layer_idx = idx
            break

    if layer_idx is None:
        raise ValueError("No layer with name '{}' within the model".format(layer_name))
    return layer_idx

In [ ]:
layer_idx = utils.find_layer_idx(model, 'aux_output')
print('layer index is:', layer_idx)
# Swap softmax with linear
model.layers[layer_idx].activation = activations.linear
model = utils.apply_modifications(model)

In [ ]:
model.save('/athena/hwlab/scratch/agp2004/MFSD2A/DNN/step_4_sensitivity_analysis/model1.h5')

class1_sal.dump('/athena/hwlab/scratch/agp2004/MFSD2A/DNN/step_4_sensitivity_analysis/class1_test.dat')
class2_sal.dump('/athena/hwlab/scratch/agp2004/MFSD2A/DNN/step_4_sensitivity_analysis/class2_test.dat')

vis_sal_class1_test = np.empty(class1_sal.shape[:3])
vis_sal_class1_test.dump('/athena/hwlab/scratch/agp2004/MFSD2A/DNN/step_4_sensitivity_analysis/vis_sal_class1_test.dat')
vis_sal_class2_test = np.empty(class2_sal.shape[:3])
vis_sal_class2_test.dump('/athena/hwlab/scratch/agp2004/MFSD2A/DNN/step_4_sensitivity_analysis/vis_sal_class2_test.dat')

In [ ]:
vis_sal_class1_test = np.load('/athena/hwlab/scratch/agp2004/MFSD2A/DNN/step_4_sensitivity_analysis/vis_sal_class1_test.dat')
vis_sal_class1_test_mean = np.sum(vis_sal_class1_test, axis=0)/vis_sal_class1_test.shape[0]

vis_sal_class2_test = np.load('/athena/hwlab/scratch/agp2004/MFSD2A/DNN/step_4_sensitivity_analysis/vis_sal_class2_test.dat')
vis_sal_class2_test_mean = np.sum(vis_sal_class2_test, axis=0)/vis_sal_class2_test.shape[0]



In [ ]:
plt.imshow(vis_sal_class1_test_mean, cmap='jet')
plt.colorbar()
plt.title('Class1 saliency average');

# plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/no_chol_sal.png', dpi=300)

In [ ]:
plt.plot(vis_sal_class1_test_mean.flatten())
plt.title('Class1 Saliency transformed')

# plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/no_chol_sal_t.png', dpi=300)

In [ ]:
out_class1 = []
for i in range(len(vis_sal_class1_test_mean.flatten())):
    if vis_sal_class1_test_mean.flatten()[i]>.6:
        out_class1.append(i+1) # vmd indices are 1-indexed

out_class1 = np.asarray(out_class1)
print(out_class1)
out_class1.shape[0]

In [ ]:
plt.imshow(vis_sal_class2_test_mean, cmap='jet')
plt.colorbar()
plt.title('Class2 saliency average');

# plt.savefig('/athena/hwlab/scratch/agp2004/', dpi=300)

In [ ]:
plt.plot(vis_sal_class2_test_mean.flatten())
plt.title('Class2 Saliency transformed')

# plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/chol_sal_t.png', dpi=300)

In [ ]:
out_class2 = []
for i in range(len(vis_sal_class2_test_mean.flatten())):
    if vis_sal_class2_test_mean.flatten()[i]>.6:
        out_class2.append(i+1) # vmd indices are 1-indexed

out_class2 = np.asarray(out_class2)
print(out_class2)
out_class2.shape[0]

In [ ]:
import MDAnalysis as mda
psf = '/athena/hwlab/scratch/agp2004/MFSD2A/DNN/data/occluded/just_protein.psf'
traj = '/athena/hwlab/scratch/agp2004/MFSD2A/DNN/data/occluded/traj_scrambled.dcd'

u = mda.Universe(psf, traj)

class1_important_resids = np.unique([u.atoms.resids[i-1] for i in out_class1])
class2_important_resids = np.unique([u.atoms.resids[i-1] for i in out_class2])
combined_important_resids = np.unique(np.concatenate((class1_important_resids, class2_important_resids)))

print('class1_important_resids')
print(class1_important_resids)

print('class2_important_resids')
print(class2_important_resids)

print('combined_important_resids')
print(combined_important_resids)



In [ ]:
index = 2680

u.atoms.resids[index-1]

In [ ]:
u.atoms.names[index-1]

In [ ]:
get_pool5_layer_output = K.function([model.layers[0].input],[model.layers[-2].output])
layer_output = get_pool5_layer_output([X_test[0].reshape(1,72,73,3)])[0][0]

In [ ]:
CHOL_test_pool5 = np.empty((CHOL_test.shape[0],2208))
for j in range(CHOL_test.shape[0]):
    print(j)
    CHOL_test_pool5[j] = get_pool5_layer_output([CHOL_test[j].reshape(1,72,73,3)])[0][0]

In [ ]:
NO_CHOL_test_pool5 = np.empty((NO_CHOL_test.shape[0],2208))
for j in range(NO_CHOL_test.shape[0]):
    print(j)
    NO_CHOL_test_pool5[j] = get_pool5_layer_output([NO_CHOL_test[j].reshape(1,72,73,3)])[0][0]

In [ ]:
# CHOL_test_pool5.dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/10/CHOL_test_pool5.dat')
# NO_CHOL_test_pool5.dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/10/NO_CHOL_test_pool5.dat')
# X_test_pool5.dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/10/X_test_pool5.dat')

CHOL_test_pool5 = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/10/CHOL_test_pool5.dat')
NO_CHOL_test_pool5 = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/10/NO_CHOL_test_pool5.dat')
X_test_pool5 = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/10/X_test_pool5.dat')

In [ ]:
X_test_pool5 = np.empty((X_test.shape[0],2208))
for j in range(X_test.shape[0]):
    print(j)
    X_test_pool5[j] = get_pool5_layer_output([X_test[j].reshape(1,72,73,3)])[0][0]

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(X_test_pool5.reshape((X_test_pool5.shape[0],-1)))

In [ ]:
pca_xtest = pca.transform(X_test_pool5.reshape((X_test_pool5.shape[0],-1)))
pca_chol = pca.transform(CHOL_test_pool5.reshape((CHOL_test_pool5.shape[0],-1)))
pca_no_chol = pca.transform(NO_CHOL_test_pool5.reshape((NO_CHOL_test_pool5.shape[0],-1)))

In [ ]:
X_test[ind].shape
Y_test[ind].shape

In [ ]:
scores = model.predict(X_test, verbose=1)
x_test_sort = np.argmax(scores, axis=1)==np.argmax(Y_test, axis=1)
chol_ind = ((np.argmax(Y_test, axis=1)==1) & x_test_sort)
no_chol_ind = ((np.argmax(Y_test, axis=1)==0) & x_test_sort)

In [ ]:
plt.scatter(pca_xtest[chol_ind,0], pca_xtest[chol_ind,1]
            , alpha=0.05, c='red', label = 'Chol')
plt.scatter(pca_xtest[no_chol_ind,0], pca_xtest[no_chol_ind,1]
            , alpha=0.05, c='blue', label='No Chol')

plt.legend()
plt.title("PCA pool5-test set, s=4ns")

plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/PCA_testset_split.png', dpi=300)

In [ ]:
np.sum(scores_sort)

In [ ]:
scores = model.predict(CHOL_test, verbose=1)
chol_ind = np.argmax(scores, axis=1)==np.argmax(chol_test, axis=1)

scores = model.predict(NO_CHOL_test, verbose=1)
no_chol_ind = np.argmax(scores, axis=1)==np.argmax(no_chol_test, axis=1)

In [ ]:
plt.scatter(pca_chol[chol_ind,0], pca_chol[chol_ind,1], alpha=0.02, c='gold', label = 'Chol')
plt.scatter(pca_no_chol[no_chol_ind,0], pca_no_chol[no_chol_ind,1], alpha=0.02, c='green', label = 'No Chol')

ind = [21796, 9048, 22317, 546, 5674, 6641, 14384]
for i in ind:
    plt.scatter(pca_no_chol[i,0], pca_no_chol[i,1], label = i)
    
# ind = [17855]
# for i in ind:
#     plt.scatter(pca_chol[i,0], pca_chol[i,1], label = i)
    
plt.legend()
plt.title("PCA pool5-held out set, s=4ns")

plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/PCA_heldset_split.png', dpi=300)

In [ ]:
plt.scatter(pca_chol[:,0], pca_chol[:,1], alpha=0.02, c='gold', label = 'Chol')
plt.scatter(pca_no_chol[:,0], pca_no_chol[:,1], alpha=0.02, c='green', label = 'No Chol')

ind = [21796, 9048, 22317, 546, 5674, 6641, 14384]
for i in ind:
    plt.scatter(pca_no_chol[i,0], pca_no_chol[i,1], label = i)
    
# ind = [17855]
# for i in ind:
#     plt.scatter(pca_chol[i,0], pca_chol[i,1], label = i)
    
plt.legend()
plt.title("PCA pool5-held out set, s=4ns")

# plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/PCA_heldset_movie.png', dpi=300)

In [ ]:
x = -6
y = 0
delta = 5

np.where((pca_no_chol[:,0] >= x-delta) & (pca_no_chol[:,0] <=x+delta) 
         & (pca_no_chol[:,1] >= y-delta) & (pca_no_chol[:,1] <=y+delta))

# np.where((pca_chol[:,0] >= x-delta) & (pca_chol[:,0] <=x+delta) 
#          & (pca_chol[:,1] >= y-delta) & (pca_chol[:,1] <=y+delta))

In [ ]:
plt.plot(pca.explained_variance_ratio_)

In [ ]:
print(np.asarray(ind)+1)

In [ ]:
scores = model.predict(X_test, verbose=1)
ind = (np.argmax(scores, axis=1)==np.argmax(Y_test, axis=1))
# X_test[ind].dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/X_test.dat')
Y_test[ind].dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/Y_test.dat')

scores = model.predict(X_train, verbose=1)
ind = (np.argmax(scores, axis=1)==np.argmax(Y_train, axis=1))
# X_train[ind].dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/X_train.dat')
Y_train[ind].dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/Y_train.dat')

scores = model.predict(X_valid, verbose=1)
ind = (np.argmax(scores, axis=1)==np.argmax(Y_valid, axis=1))
# X_valid[ind].dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/X_valid.dat')
Y_valid[ind].dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/Y_valid.dat')

scores = model.predict(CHOL_test, verbose=1)
ind = (np.argmax(scores, axis=1)==np.argmax(chol_test, axis=1))
# CHOL_test[ind].dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/CHOL_test.dat')
chol_test[ind].dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/chol_test.dat')

scores = model.predict(NO_CHOL_test, verbose=1)
ind = (np.argmax(scores, axis=1)==np.argmax(no_chol_test, axis=1))
# NO_CHOL_test[ind].dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/NO_CHOL_test.dat')
no_chol_test[ind].dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/no_chol_test.dat')

In [ ]:
X_test = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/X_test.dat')
X_train = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/X_train.dat')
X_valid = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/X_valid.dat')
Y_test = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/Y_test.dat')
Y_train = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/Y_train.dat')
Y_valid = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/Y_valid.dat')

CHOL_test = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/CHOL_test.dat')
NO_CHOL_test = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/NO_CHOL_test.dat')
chol_test = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/chol_test.dat')
no_chol_test = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/no_chol_test.dat')

In [ ]:
def fitmodel(X_train, Y_train, X_valid, Y_valid):
    if __name__ == '__main__':

        img_rows, img_cols =  X_train.shape[1], X_train.shape[2] # Resolution of inputs
        channel = 3
        num_classes = 2
        batch_size = 6
        nb_epoch = 100
        lr=1e-3

        # Load our model
        model = densenet161_model(img_rows=img_rows, img_cols=img_cols, color_type=channel, num_classes=num_classes)

        def lr_schedule(epoch, lr):
            return lr * (0.1 ** int(epoch / 10))
    #     LearningRateScheduler(lr_schedule)

#         checkpointer = ModelCheckpoint(filepath='/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/weights_n%s.hdf5' %n, monitor='val_loss', verbose=1, save_weights_only=True, save_best_only=True)
#         # Start Fine-tuning
#         history = model.fit(X_train, Y_train, validation_data=(X_valid, Y_valid), epochs=nb_epoch, batch_size=batch_size, callbacks=[checkpointer])
        return model

model = fitmodel(X_train, Y_train, X_valid, Y_valid)

In [ ]:
model.load_weights('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/weights_n%s.hdf5' %n)

no_chol_ind = np.where(np.argmax(Y_test, axis=1)==0)
scores = model.predict(X_test[no_chol_ind], verbose=1)
accuracy = 100*np.sum(np.argmax(scores, axis=1)==np.argmax(Y_test[no_chol_ind], axis=1))/Y_test[no_chol_ind].shape[0]
print("NO_CHOL test set accuracy is    %",accuracy)

chol_ind = np.where(np.argmax(Y_test, axis=1)==1)
scores = model.predict(X_test[chol_ind], verbose=1)
accuracy = 100*np.sum(np.argmax(scores, axis=1)==np.argmax(Y_test[chol_ind], axis=1))/Y_test[chol_ind].shape[0]
print("CHOL test set accuracy is    %",accuracy)

scores = model.predict(NO_CHOL_test, verbose=1)
accuracy = 100*np.sum(np.argmax(scores, axis=1)==np.argmax(no_chol_test, axis=1))/no_chol_test.shape[0]
print("NO_CHOL held out accuracy is    %",accuracy)

scores = model.predict(CHOL_test, verbose=1)
accuracy = 100*np.sum(np.argmax(scores, axis=1)==np.argmax(chol_test, axis=1))/chol_test.shape[0]
print("CHOL held out accuracy is    %",accuracy)
                                                     

In [ ]:
scores = model.predict(NO_CHOL_test, verbose=1)
no_chol_sal = NO_CHOL_test[(np.argmax(scores, axis=1)==0)][:1000]
print(no_chol_sal.shape)

scores = model.predict(CHOL_test, verbose=1)
chol_sal = CHOL_test[(np.argmax(scores, axis=1)==1)][:1000]
print(chol_sal.shape)

In [ ]:
layer_idx = utils.find_layer_idx(model, 'aux_output')

# Swap softmax with linear
model.layers[layer_idx].activation = activations.linear
model = utils.apply_modifications(model)

In [ ]:
model.save('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/XYZ/model1.h5')

no_chol_sal.dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/XYZ/no_chol_test.dat')
chol_sal.dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/XYZ/chol_test.dat')

vis_sal_no_chol_test = np.empty(no_chol_sal.shape[:3])
vis_sal_no_chol_test.dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/XYZ/vis_sal_no_chol_test.dat')
vis_sal_chol_test = np.empty(chol_sal.shape[:3])
vis_sal_chol_test.dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/XYZ/vis_sal_chol_test.dat')

In [ ]:
vis_sal_no_chol_test = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/XYZ/vis_sal_no_chol_test.dat')
vis_sal_no_chol_test_mean = np.sum(vis_sal_no_chol_test, axis=0)/840

vis_sal_chol_test = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/XYZ/vis_sal_chol_test.dat')
vis_sal_chol_test_mean = np.sum(vis_sal_chol_test, axis=0)/840

In [ ]:
plt.imshow(vis_sal_no_chol_test_mean, cmap='jet')
plt.colorbar()
plt.title('No Cholesterol saliency average');

plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/figures/no_chol_sal_correct.png', format='png', dpi=300)

In [ ]:
plt.plot(vis_sal_no_chol_test_mean.flatten())
plt.title('No Cholesterol Saliency transformed')

# plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/no_chol_sal_t.png', dpi=300)

In [ ]:
out_no_chol = []
for i in range(len(vis_sal_no_chol_test_mean.flatten())):
    if vis_sal_no_chol_test_mean.flatten()[i]>.25:
        out_no_chol.append(i+1)

out_no_chol = np.asarray(out_no_chol).astype(int)
print(out_no_chol)
out_no_chol.shape[0]

np.savetxt("/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/out_no_chol.txt", out_no_chol, fmt='%i', newline=" ")

In [ ]:
plt.imshow(vis_sal_chol_test_mean, cmap='jet')
plt.colorbar()
plt.title('Cholesterol saliency average');

plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/figures/chol_sal_correct.png', format='png', dpi=300)

In [ ]:
plt.plot(vis_sal_chol_test_mean.flatten())
plt.title('Cholesterol Saliency transformed')

# plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/chol_sal_t.png', dpi=300)

In [ ]:
out_chol = []
for i in range(len(vis_sal_chol_test_mean.flatten())):
    if vis_sal_chol_test_mean.flatten()[i]>.28:
        out_chol.append(i+1)

out_chol = np.asarray(out_chol).astype(int)
print(out_chol)
out_chol.shape[0]
np.savetxt("/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/out_chol.txt", out_chol, fmt='%i', newline=" ")

In [ ]:
get_pool5_layer_output = K.function([model.layers[0].input],[model.layers[-2].output])
layer_output = get_pool5_layer_output([X_test[0].reshape(1,72,73,3)])[0][0]



In [ ]:
NO_CHOL_test_pool5 = np.empty((NO_CHOL_test.shape[0],2208))
for j in range(NO_CHOL_test.shape[0]):
    print(j)
    NO_CHOL_test_pool5[j] = get_pool5_layer_output([NO_CHOL_test[j].reshape(1,72,73,3)])[0][0]

In [ ]:
CHOL_test_pool5 = np.empty((CHOL_test.shape[0],2208))
for j in range(CHOL_test.shape[0]):
    print(j)
    CHOL_test_pool5[j] = get_pool5_layer_output([CHOL_test[j].reshape(1,72,73,3)])[0][0]

In [ ]:
X_test_pool5 = np.empty((X_test.shape[0],2208))
for j in range(X_test.shape[0]):
    print(j)
    X_test_pool5[j] = get_pool5_layer_output([X_test[j].reshape(1,72,73,3)])[0][0]

In [ ]:
# CHOL_test_pool5.dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/CHOL_test_pool5.dat')
# NO_CHOL_test_pool5.dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/NO_CHOL_test_pool5.dat')
# X_test_pool5.dump('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/X_test_pool5.dat')

CHOL_test_pool5 = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/CHOL_test_pool5.dat')
NO_CHOL_test_pool5 = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/NO_CHOL_test_pool5.dat')
X_test_pool5 = np.load('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/correct/X_test_pool5.dat')

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(X_test_pool5.reshape((X_test_pool5.shape[0],-1)))

In [ ]:
pca_xtest = pca.transform(X_test_pool5.reshape((X_test_pool5.shape[0],-1)))
pca_chol = pca.transform(CHOL_test_pool5.reshape((CHOL_test_pool5.shape[0],-1)))
pca_no_chol = pca.transform(NO_CHOL_test_pool5.reshape((NO_CHOL_test_pool5.shape[0],-1)))

In [ ]:
plt.scatter(pca_xtest[chol_ind,0], pca_xtest[chol_ind,1]
            , alpha=0.05, c='red', label = 'Chol')
plt.scatter(pca_xtest[no_chol_ind,0], pca_xtest[no_chol_ind,1]
            , alpha=0.05, c='blue', label='No Chol')

plt.legend()
plt.title("PCA pool5-test set, s=4ns")

plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/PCA_testset_correct.png', dpi=300)

In [ ]:
plt.scatter(pca_chol[:,0], pca_chol[:,1], alpha=0.02, c='gold', label = 'Chol')
plt.scatter(pca_no_chol[:,0], pca_no_chol[:,1], alpha=0.02, c='green', label = 'No Chol')

# ind = [21796, 9048, 22317, 546, 5674, 6641, 14384]
# for i in ind:
#     plt.scatter(pca_no_chol[i,0], pca_no_chol[i,1], label = i)
    
# ind = [17855]
# for i in ind:
#     plt.scatter(pca_chol[i,0], pca_chol[i,1], label = i)
    
plt.legend()
plt.title("PCA pool5-held out set, s=4ns")

plt.savefig('/athena/hwlab/scratch/agp2004/Opsin/ML_tests/figures/PCA_heldset_correct.png', dpi=300)

In [ ]:
h5f1 = h5py.File('/athena/hwlab/scratch/agp2004/Opsin/temp/NO_CHOL_XYZ.dat','r')
NO_CHOL = h5f1['dataset_2'][:]
h5f1.close()

h5f2 = h5py.File('/athena/hwlab/scratch/agp2004/Opsin/temp/CHOL_XYZ.dat','r')
CHOL = h5f2['dataset_1'][:]
h5f2.close()



In [ ]:
no_chol_labels = np.zeros(NO_CHOL.shape[0])
chol_labels = np.ones(CHOL.shape[0])



In [ ]:
CHOL = train_datagen.standardize(CHOL)
NO_CHOL = train_datagen.standardize(NO_CHOL)

chol_labels = np_utils.to_categorical(chol_labels, num_classes)
no_chol_labels = np_utils.to_categorical(no_chol_labels, num_classes)



In [ ]:
scores = model.predict(CHOL, verbose=1)
accuracy = 100*np.sum(np.argmax(scores, axis=1)==np.argmax(chol_labels, axis=1))/chol_labels.shape[0]
print("CHOL test set accuracy is    %",accuracy)



In [ ]:
scores = model.predict(NO_CHOL, verbose=1)
accuracy = 100*np.sum(np.argmax(scores, axis=1)==np.argmax(no_chol_labels, axis=1))/no_chol_labels.shape[0]
print("NO_CHOL test set accuracy is    %",accuracy)



In [ ]:
NO_CHOL_pool5 = np.empty((NO_CHOL.shape[0],2208))
for j in range(NO_CHOL.shape[0]):
    print(j)
    NO_CHOL_pool5[j] = get_pool5_layer_output([NO_CHOL[j].reshape(1,72,73,3)])[0][0]
    
h5f1 = h5py.File('/athena/hwlab/scratch/agp2004/Opsin/temp/NO_CHOL_pool5.dat', 'w')
h5f1.create_dataset('dataset_3', data=NO_CHOL_pool5)
h5f1.close()



In [ ]:
CHOL_pool5 = np.empty((CHOL.shape[0],2208))
for j in range(CHOL.shape[0]):
    print(j)
    CHOL_pool5[j] = get_pool5_layer_output([CHOL[j].reshape(1,72,73,3)])[0][0]
    
h5f1 = h5py.File('/athena/hwlab/scratch/agp2004/Opsin/temp/CHOL_pool5.dat', 'w')
h5f1.create_dataset('dataset_4', data=CHOL_pool5)
h5f1.close()
    